# Classificação de Cogumelos com Regras de Classificação

**Objetivo:** Prever se um cogumelo é **venenoso (p)** ou **comestível (e)** com base em suas características físicas, utilizando o algoritmo **1R (One Rule)**.

**Dataset:** [Mushroom Dataset (UCI)](https://www.kaggle.com/datasets/uciml/mushroom-classification).

---

### O que é o algoritmo 1R?

O **1R (One Rule)** é um dos algoritmos de classificação mais simples que existem. Ele gera **uma única regra** baseada na **melhor feature**, aquela que sozinha comete o menor número de erros ao classificar os exemplos.

**Como funciona:**
1. Para cada feature, cria uma regra simples: cada valor da feature é mapeado para a classe mais frequente
2. Calcula o erro de cada feature
3. Escolhe a feature com **menor erro** como a regra final

**Vantagens:**
- Extremamente simples e interpretável
- Rápido para treinar
- Serve como baseline

In [31]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

---
## 1. Carregamento dos Dados

In [32]:
df = pd.read_csv('../data.csv')

print(f'Colunas lidas: {list(df.columns)}\n')
df

Colunas lidas: ['class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']



,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,e,k,s,n,f,n,a,c,b,y,...,s,o,o,p,o,o,p,b,c,l
8120,e,x,s,n,f,n,a,c,b,y,...,s,o,o,p,n,o,p,b,v,l
8121,e,f,s,n,f,n,a,c,b,n,...,s,o,o,p,o,o,p,b,c,l
8122,p,k,y,n,f,y,f,c,n,b,...,k,w,w,p,w,o,e,w,v,l


---
## 2. Limpeza e Preparação dos Dados

In [33]:

FEATURES = [col for col in df.columns if col != 'class']
TARGET   = 'class'

print(f'Serão no total {len(FEATURES)} features utilizadas')

Serão no total 22 features utilizadas


### 2.1 Distribuição do Target

In [34]:
df[TARGET] = df[TARGET].astype(str).str.strip().str.lower()
 
df = df.dropna(subset=FEATURES + [TARGET]).copy()

display(df.head())

print('Distribuição do target:')
print(df[TARGET].value_counts())

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


Distribuição do target:
class
e    4208
p    3916
Name: count, dtype: int64


---
## 3. Separação em Treino e Teste

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df[TARGET],
    test_size=0.2,
    random_state=42,
    stratify=df[TARGET]
)

print(f'Treino: {len(X_train)/len(df)*100:.0f}% = {len(X_train)} amostras')
print(f'Teste: {len(X_test)/len(df)*100:.0f}% = {len(X_test)} amostras')

Treino: 80% = 6499 amostras
Teste: 20% = 1625 amostras


---
## 4. Implementação do Algoritmo 1R

In [36]:
def one_r(X_train, y_train):
    b_feature = None
    b_rule = {}
    l_err = float('inf')

    for feature in X_train.columns:
        rule = (
            X_train.join(y_train)
            .groupby(feature)[TARGET]
            .agg(lambda x: x.value_counts().idxmax())
            .to_dict()
        )

        y_pred = X_train[feature].map(rule)
        errs = (y_pred != y_train).sum()

        if errs < l_err:
            l_err     = errs
            b_feature = feature
            b_rule   = rule

    return b_feature, b_rule, l_err

b_feature, b_rule, err_train = one_r(X_train, y_train)

print(f'Melhor feature : {b_feature}')
print(f'Erros no treino: {err_train}/{len(X_train)}\n')
print('Regra gerada:')
for v, c in sorted(b_rule.items()):
    print(f'Se {b_feature} = "{v}"  →  {c}')

classe_default = y_train.value_counts().idxmax()
y_pred = X_test[b_feature].map(b_rule).fillna(classe_default)

Melhor feature : odor
Erros no treino: 97/6499

Regra gerada:
Se odor = "a"  →  e
Se odor = "c"  →  p
Se odor = "f"  →  p
Se odor = "l"  →  e
Se odor = "m"  →  p
Se odor = "n"  →  e
Se odor = "p"  →  p
Se odor = "s"  →  p
Se odor = "y"  →  p


---
## 6. Avaliação do Modelo

In [37]:
print(f"Acurácia: {accuracy_score(y_test, y_pred):.2%}")

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, target_names=['Comestível (e)', 'Venenoso (p)']))

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred, labels=['e', 'p']))

Acurácia: 98.58%

Relatório de classificação:
                precision    recall  f1-score   support

Comestível (e)       0.97      1.00      0.99       842
  Venenoso (p)       1.00      0.97      0.99       783

      accuracy                           0.99      1625
     macro avg       0.99      0.99      0.99      1625
  weighted avg       0.99      0.99      0.99      1625

Matriz de confusão:
[[842   0]
 [ 23 760]]
